In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Setup Paths and Parameters (Reverted for LeNet) ---

data_path = 'mnist-pngs-main/' 
train_path = data_path + 'train'
test_path = data_path + 'test'

# --- CHANGES FOR LENET-5 ---
# LeNet-5 was designed for 32x32 images, but works perfectly on 28x28.
# We will use 28x28 and let the model handle padding.
BATCH_SIZE = 32
IMG_HEIGHT = 28
IMG_WIDTH = 28

# --- 2. Load the Datasets ---
# We are back to loading 28x28 grayscale images.

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_path,
    labels='inferred',
    label_mode='int',
    color_mode='grayscale', # Load as grayscale
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_path,
    labels='inferred',
    label_mode='int',
    color_mode='grayscale',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

class_names = train_dataset.class_names
print(f"Class names: {class_names}")

# --- 3. Prepare the Dataset for LeNet-5 ---

# We just need to normalize the pixel values to the [0, 1] range.
def normalize(image, label):
    return tf.cast(image, tf.float32) / 255.0, label

# Apply the normalization function
train_dataset = train_dataset.map(normalize)
test_dataset = test_dataset.map(normalize)

# Optimize performance with caching and prefetching
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

print("\n--- Inspecting One Preprocessed Batch ---")
for images, labels in train_dataset.take(1):
    print(f"Images batch shape: {images.shape}") # Should be (32, 28, 28, 1)
    print(f"Labels batch shape: {labels.shape}")
    break

# --- 4. Build the LeNet-5 Model from Scratch ---

# The LeNet-5 architecture has a specific structure:
# Input -> Conv -> Pool -> Conv -> Pool -> Flatten -> Dense -> Dense -> Output

model = tf.keras.Sequential([
    # Input layer for our 28x28 grayscale images
    tf.keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 1)),
    
    # C1: First Convolutional Layer
    # 6 filters of size 5x5. Activation: tanh (classic LeNet used tanh/sigmoid)
    # Padding='same' makes it work nicely with 28x28 inputs.
    tf.keras.layers.Conv2D(6, kernel_size=(5, 5), activation='tanh', padding='same'),
    
    # S2: Subsampling (Pooling) Layer
    # Average pooling of size 2x2 with a stride of 2.
    tf.keras.layers.AveragePooling2D(pool_size=(2, 2), strides=2),
    
    # C3: Second Convolutional Layer
    # 16 filters of size 5x5.
    tf.keras.layers.Conv2D(16, kernel_size=(5, 5), activation='tanh'),
    
    # S4: Second Subsampling (Pooling) Layer
    tf.keras.layers.AveragePooling2D(pool_size=(2, 2), strides=2),

    # C5: Third Convolutional Layer (often implemented as a Dense layer)
    # This is a convolution with 120 filters of size 5x5, which results in a 1x1x120 output.
    # It's equivalent to flattening and adding a Dense layer.
    tf.keras.layers.Conv2D(120, kernel_size=(5, 5), activation='tanh'),
    
    # Flatten the output to feed into the dense layers
    tf.keras.layers.Flatten(),
    
    # F6: First Fully-Connected (Dense) Layer
    # 84 neurons.
    tf.keras.layers.Dense(84, activation='tanh'),
    
    # Output Layer
    # 10 neurons (for 10 digits) with softmax for classification.
    tf.keras.layers.Dense(10, activation='softmax')
])


# --- 5. Compile and Train the Model ---

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

print("\n--- LeNet-5 Model Summary ---")
model.summary()

print("\n--- Starting Model Training ---")
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=10 # LeNet is small and trains fast, we can use more epochs
)

# --- 6. Evaluate the Model ---
print("\n--- Evaluating Model on Test Data ---")
loss, accuracy = model.evaluate(test_dataset)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

Found 55843 files belonging to 10 classes.
Found 9684 files belonging to 10 classes.
Class names: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

--- Inspecting One Preprocessed Batch ---
Images batch shape: (32, 28, 28, 1)
Labels batch shape: (32,)

--- LeNet-5 Model Summary ---


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 14, 14, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 5, 5, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 1, 1, 120)      │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)


--- Starting Model Training ---
Epoch 1/10
1737/1746 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.8790 - loss: 0.4083

KeyboardInterrupt: 